# Lab 08 — Object Detection Using Faster R-CNN
**Course:** Computer Vision (AI-4002)  
**Student:** ____________________  
**Roll number:** ____________________

## Objective
Implement region-based object detection on a dataset other than BCCD, including bounding-box annotations, data exploration, training, validation, evaluation, and inference. This notebook uses a **self-generated synthetic geometric-sign dataset** containing circles, squares and triangles, and trains a real **Faster R-CNN** detector with a lightweight CNN backbone.

> The dataset is synthetic and intended for an end-to-end educational demonstration. These results must not be interpreted as performance on real-world traffic-sign images.

## 1. Environment and reproducibility
The notebook runs locally from the project folder. Installation (once):

```bash
python -m pip install -r requirements.txt
```

The default device is CPU; CUDA can be used automatically if installed.

In [ ]:
import sys, subprocess, json
from pathlib import Path
from IPython.display import display, Image, Markdown
import torch, torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
ROOT = Path.cwd()
print('Project directory:', ROOT)

## 2. Dataset generation and inspection
Use a fixed seed of 42 to create **150 training**, **30 validation**, and **30 held-out testing** images. Each image is 128×128 pixels with 1–3 objects on lightly varied backgrounds. CSV annotations store `filename, class, xmin, ymin, xmax, ymax` as pixel coordinates.

In [ ]:
result = subprocess.run([sys.executable, 'lab08.py', 'generate', '--seed', '42', '--n-train', '150', '--n-val', '30', '--n-test', '30'], capture_output=True, text=True, check=True)
print(result.stdout)
print((ROOT/'data'/'train.csv').read_text().splitlines()[:5])

In [ ]:
display(Image(filename=str(ROOT/'figures'/'ground_truth_examples.png')))
display(Image(filename=str(ROOT/'figures'/'class_distribution.png')))

## 3. Faster R-CNN architecture
The implementation has:
1. **CNN backbone** for shared image features (4 convolutions).
2. **Region Proposal Network (RPN)** to propose candidate regions with anchors.
3. **RoI Align** to obtain fixed-size region features.
4. **Classification and bounding-box regression** to label and localize objects.

Unlike original R-CNN, the CNN feature extraction is shared and the RPN is learned jointly. We train from scratch; no downloaded pretrained model weights are needed.

`0 = background`, `1 = circle`, `2 = square`, `3 = triangle`. The four-part loss combines RPN objectness, RPN box regression, RoI classification and RoI box regression.

In [ ]:
from lab08 import build_model, ObjectDataset, CLASSES
model=build_model()
print('Classes:', CLASSES)
print('Parameters:', sum(p.numel() for p in model.parameters()))
image, target = ObjectDataset('train')[0]
print('Image tensor:', tuple(image.shape))
print('Example targets:', {k: v.tolist() for k,v in target.items() if k in ('boxes','labels')})

## 4. Train the model
We use Adam with learning rate 0.001, batch size 4, a seeded shuffle, horizontal-flip augmentation and 12 epochs. The highest validation mAP@0.50 checkpoint is saved as `outputs/best_model.pth`. The validation set is used to select the model; the test set is untouched until the next section.

In [ ]:
result = subprocess.run([sys.executable, '-u', 'lab08.py', 'train', '--seed', '42', '--epochs', '12', '--batch-size', '4', '--lr', '0.001'], capture_output=True, text=True, check=True)
print(result.stdout)
print(json.dumps(json.loads((ROOT/'outputs'/'training_summary.json').read_text()), indent=2))

In [ ]:
display(Image(filename=str(ROOT/'figures'/'training_curves.png')))

## 5. Evaluation on held-out data
**IoU** is the area of overlap divided by union between a predicted box and its ground-truth box. At **IoU ≥ 0.50**, a class-correct, one-to-one matched prediction is a true positive. A false positive is an incorrect/unmatched prediction. A false negative is a missed ground-truth object.

**AP50** integrates the class precision–recall curve; **mAP50** averages AP50 across the three classes. We also report mAP75, precision, recall and F1 at a **0.35 confidence threshold**.

In [ ]:
result = subprocess.run([sys.executable, 'lab08.py', 'test', '--threshold', '0.35'], capture_output=True, text=True, check=True)
print(result.stdout)
metrics=json.loads((ROOT/'outputs'/'test_metrics.json').read_text())
print('Selected summary:', {k:metrics[k] for k in ('mAP50','mAP75','precision_at_threshold','recall_at_threshold','f1_at_threshold','mean_matched_iou')})

In [ ]:
display(Image(filename=str(ROOT/'figures'/'test_predictions.png')))

## 6. Inference on another image
Run the trained model against any generated test image; it exports both an annotated PNG and a machine-readable JSON prediction. You can replace the path with your own PNG/JPG. Because this model is trained on simple simulated signs, it is not expected to generalize to arbitrary real photos.

In [ ]:
result = subprocess.run([sys.executable, 'lab08.py', 'predict', '--image', 'data/test/test_0001.png', '--threshold', '0.35'], capture_output=True, text=True, check=True)
print(result.stdout)
display(Image(filename=str(ROOT/'outputs'/'test_0001_prediction.png')))

## 7. Conclusions and limitations
The complete pipeline detects and localizes objects using a Faster R-CNN network rather than a simple image classifier. The data generator creates independent train/validation/test images with exact ground-truth bounding boxes, and all artifacts are saved to this project.

**Limitations:** Synthetic objects are color-distinct, backgrounds are simple and there is little occlusion, so a perfect result on this test set is not a reliable estimate of performance on natural images. Future work includes adapting this pipeline to an independently labeled real detection dataset and measuring COCO-style mAP across multiple IoU thresholds.

**References:** Lab 08 Manual, sections 2–6; PyTorch/TorchVision object detection documentation.